In [72]:
import pandas as pd
from datetime import timedelta, date
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
from sqlalchemy import create_engine, text

In [2]:
trash_hauler_report = pd.read_csv('../trash_hauler_report.csv')
trash_hauler_report

,Request Number,Date Opened,Request,Description,Incident Address,Zip Code,Trash Hauler,Trash Route,Council District,State Plan X,State Plan Y
0,25270,11/1/2017,Trash - Backdoor,"house with the wheel chair ramp, they share dr...",3817 Crouch Dr,37207.0,RED RIVER,3205,2.0,1727970.412,686779.4781
1,25274,11/1/2017,Trash - Curbside/Alley Missed Pickup,Curb/Trash miss Tuesday.,4028 Clarksville Pike,37218.0,RED RIVER,4202,1.0,1721259.366,685444.7996
2,25276,11/1/2017,Trash - Curbside/Alley Missed Pickup,Curb/trash miss Tuesday.,6528 Thunderbird Dr,37209.0,RED RIVER,4205,20.0,1707026.753,659887.4716
3,25307,11/1/2017,Trash - Curbside/Alley Missed Pickup,missed,2603 old matthews rd,37207.0,WASTE IND,2206,2.0,1735691.771,685027.2459
4,25312,11/1/2017,Trash - Curbside/Alley Missed Pickup,Missed the even side of the road.,604 croley dr,37209.0,RED RIVER,4203,20.0,1710185.772,664205.1011
...,...,...,...,...,...,...,...,...,...,...,...
20221,267125,11/1/2019,Trash - Curbside/Alley Missed Pickup,MISSED...NEIGHBORS MISSED,2731 Murfreesboro Pike,37013.0,RED RIVER,4502,32.0,1781137.263,632448.5511
20222,267126,11/1/2019,Trash - Curbside/Alley Missed Pickup,entire alley,"1621 Long Ave, Nashville, TN 37206, United States",37206.0,METRO,9508,6.0,1749711.399,669201.6016
20223,267130,11/1/2019,Trash - Curbside/Alley Missed Pickup,missed several,"2943 Windemere Cir, Nashville, TN 37214, Unite...",37214.0,RED RIVER,1502,15.0,1770293.388,674936.3038
20224,267134,11/1/2019,Trash - Curbside/Alley Missed Pickup,Caller stated trash was missed & were only pic...,"3325 Murfreesboro Pike, Nashville, TN 37013, U...",37013.0,RED RIVER,4502,32.0,1785224.998,627146.4002


In [3]:
trash_hauler_report.columns

Index(['Request Number', 'Date Opened', 'Request ', 'Description',
       'Incident Address', 'Zip Code', 'Trash Hauler', 'Trash Route',
       'Council District', 'State Plan X', 'State Plan Y'],
      dtype='object')

In [4]:
trash_hauler_report['Request '].value_counts()

Request 
Trash - Curbside/Alley Missed Pickup    15028
Trash - Backdoor                         2629
Trash Collection Complaint               2312
Damage to Property                        257
Name: count, dtype: int64

In [5]:
trash_hauler_report['Incident Address'].value_counts()

Incident Address
5135 Hickory Hollow Pkwy                                      21
3710 N NATCHEZ CT                                             20
6007 Obrien Ave, Nashville, TN 37209, United States           19
12546 Old Hickory Blvd, Nashville, TN 37013, United States    19
802 Crescent Rd, Nashville, TN 37205, United States           18
                                                              ..
2924 Fernbrook Ln, Nashville, TN 37214, United States          1
356 Kingview Dr                                                1
1414 Old Hickory Blvd                                          1
117 Lafayette Ct                                               1
1904 MeHarry Blvd                                              1
Name: count, Length: 14121, dtype: int64

In [6]:
trash_hauler_report.columns = ['request_num', 'date', 'request', 'description',
       'address', 'zip', 'trash_hauler', 'trash_route',
       'council_district', 'state_plan_x', 'state_plan_y']
trash_hauler_report.columns

Index(['request_num', 'date', 'request', 'description', 'address', 'zip',
       'trash_hauler', 'trash_route', 'council_district', 'state_plan_x',
       'state_plan_y'],
      dtype='object')

In [7]:
missed = trash_hauler_report[trash_hauler_report.request.isin(['Trash - Curbside/Alley Missed Pickup'])]
missed

,request_num,date,request,description,address,zip,trash_hauler,trash_route,council_district,state_plan_x,state_plan_y
1,25274,11/1/2017,Trash - Curbside/Alley Missed Pickup,Curb/Trash miss Tuesday.,4028 Clarksville Pike,37218.0,RED RIVER,4202,1.0,1721259.366,685444.7996
2,25276,11/1/2017,Trash - Curbside/Alley Missed Pickup,Curb/trash miss Tuesday.,6528 Thunderbird Dr,37209.0,RED RIVER,4205,20.0,1707026.753,659887.4716
3,25307,11/1/2017,Trash - Curbside/Alley Missed Pickup,missed,2603 old matthews rd,37207.0,WASTE IND,2206,2.0,1735691.771,685027.2459
4,25312,11/1/2017,Trash - Curbside/Alley Missed Pickup,Missed the even side of the road.,604 croley dr,37209.0,RED RIVER,4203,20.0,1710185.772,664205.1011
8,25330,11/1/2017,Trash - Curbside/Alley Missed Pickup,Missed.,4484 Lavergne Couchville Pike,37013.0,RED RIVER,4210,33.0,1794533.514,618749.3427
...,...,...,...,...,...,...,...,...,...,...,...
20221,267125,11/1/2019,Trash - Curbside/Alley Missed Pickup,MISSED...NEIGHBORS MISSED,2731 Murfreesboro Pike,37013.0,RED RIVER,4502,32.0,1781137.263,632448.5511
20222,267126,11/1/2019,Trash - Curbside/Alley Missed Pickup,entire alley,"1621 Long Ave, Nashville, TN 37206, United States",37206.0,METRO,9508,6.0,1749711.399,669201.6016
20223,267130,11/1/2019,Trash - Curbside/Alley Missed Pickup,missed several,"2943 Windemere Cir, Nashville, TN 37214, Unite...",37214.0,RED RIVER,1502,15.0,1770293.388,674936.3038
20224,267134,11/1/2019,Trash - Curbside/Alley Missed Pickup,Caller stated trash was missed & were only pic...,"3325 Murfreesboro Pike, Nashville, TN 37013, U...",37013.0,RED RIVER,4502,32.0,1785224.998,627146.4002


In [8]:
searchfor = ['miss', 'didn\'t pick']
missed2 = trash_hauler_report[trash_hauler_report.description.str.contains('|'.join(searchfor), case=False).fillna(False)]
missed2

C:\Users\erics\AppData\Local\Temp\ipykernel_40192\437307834.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  missed2 = trash_hauler_report[trash_hauler_report.description.str.contains('|'.join(searchfor), case=False).fillna(False)]


,request_num,date,request,description,address,zip,trash_hauler,trash_route,council_district,state_plan_x,state_plan_y
1,25274,11/1/2017,Trash - Curbside/Alley Missed Pickup,Curb/Trash miss Tuesday.,4028 Clarksville Pike,37218.0,RED RIVER,4202,1.0,1721259.366,685444.7996
2,25276,11/1/2017,Trash - Curbside/Alley Missed Pickup,Curb/trash miss Tuesday.,6528 Thunderbird Dr,37209.0,RED RIVER,4205,20.0,1707026.753,659887.4716
3,25307,11/1/2017,Trash - Curbside/Alley Missed Pickup,missed,2603 old matthews rd,37207.0,WASTE IND,2206,2.0,1735691.771,685027.2459
4,25312,11/1/2017,Trash - Curbside/Alley Missed Pickup,Missed the even side of the road.,604 croley dr,37209.0,RED RIVER,4203,20.0,1710185.772,664205.1011
7,25327,11/1/2017,Trash Collection Complaint,"Trash out on time, miss again Tuesday. ALLEY",1816 Jo Johnston Ave,37203.0,METRO,9208,21.0,1731459.367,666013.6012
...,...,...,...,...,...,...,...,...,...,...,...
20220,267121,11/1/2019,Trash - Curbside/Alley Missed Pickup,missed,"2709 Crestdale Dr, Nashville, TN 37214, United...",37214.0,RED RIVER,1502,15.0,1770240.199,676334.3993
20221,267125,11/1/2019,Trash - Curbside/Alley Missed Pickup,MISSED...NEIGHBORS MISSED,2731 Murfreesboro Pike,37013.0,RED RIVER,4502,32.0,1781137.263,632448.5511
20223,267130,11/1/2019,Trash - Curbside/Alley Missed Pickup,missed several,"2943 Windemere Cir, Nashville, TN 37214, Unite...",37214.0,RED RIVER,1502,15.0,1770293.388,674936.3038
20224,267134,11/1/2019,Trash - Curbside/Alley Missed Pickup,Caller stated trash was missed & were only pic...,"3325 Murfreesboro Pike, Nashville, TN 37013, U...",37013.0,RED RIVER,4502,32.0,1785224.998,627146.4002


In [9]:
missed_pickups = pd.concat([missed, missed2]).drop_duplicates()

In [10]:
missed_pickups.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17743 entries, 1 to 20205
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   request_num       17743 non-null  int64  
 1   date              17743 non-null  object 
 2   request           17743 non-null  object 
 3   description       17714 non-null  object 
 4   address           17734 non-null  object 
 5   zip               17682 non-null  float64
 6   trash_hauler      16995 non-null  object 
 7   trash_route       16976 non-null  object 
 8   council_district  17702 non-null  float64
 9   state_plan_x      17718 non-null  float64
 10  state_plan_y      17718 non-null  float64
dtypes: float64(4), int64(1), object(6)
memory usage: 1.6+ MB


In [11]:
missed_pickups = missed_pickups.dropna(subset=['address'])
missed_pickups.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17734 entries, 1 to 20205
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   request_num       17734 non-null  int64  
 1   date              17734 non-null  object 
 2   request           17734 non-null  object 
 3   description       17705 non-null  object 
 4   address           17734 non-null  object 
 5   zip               17680 non-null  float64
 6   trash_hauler      16993 non-null  object 
 7   trash_route       16974 non-null  object 
 8   council_district  17700 non-null  float64
 9   state_plan_x      17716 non-null  float64
 10  state_plan_y      17716 non-null  float64
dtypes: float64(4), int64(1), object(6)
memory usage: 1.6+ MB


In [12]:
for i, r in missed_pickups.iterrows():
    if ('Nashville' in r.address) or ('nashville' in r.address):
        ## looking for any characters preceding ' Nashville' or ' nashville', followed by any number of other characters
        missed_pickups.loc[i, 'address'] = re.search(r'(.*),*\s?[nN]ashville.*', r['address']).group(1).replace(',', '').replace('.', '').strip().title()
    else:
        missed_pickups.loc[i, 'address'] = r.address.replace(',', '').replace('.', '').strip().title()

In [13]:
missed_pickups.head()

,request_num,date,request,description,address,zip,trash_hauler,trash_route,council_district,state_plan_x,state_plan_y
1,25274,11/1/2017,Trash - Curbside/Alley Missed Pickup,Curb/Trash miss Tuesday.,4028 Clarksville Pike,37218.0,RED RIVER,4202,1.0,1721259.366,685444.7996
2,25276,11/1/2017,Trash - Curbside/Alley Missed Pickup,Curb/trash miss Tuesday.,6528 Thunderbird Dr,37209.0,RED RIVER,4205,20.0,1707026.753,659887.4716
3,25307,11/1/2017,Trash - Curbside/Alley Missed Pickup,missed,2603 Old Matthews Rd,37207.0,WASTE IND,2206,2.0,1735691.771,685027.2459
4,25312,11/1/2017,Trash - Curbside/Alley Missed Pickup,Missed the even side of the road.,604 Croley Dr,37209.0,RED RIVER,4203,20.0,1710185.772,664205.1011
8,25330,11/1/2017,Trash - Curbside/Alley Missed Pickup,Missed.,4484 Lavergne Couchville Pike,37013.0,RED RIVER,4210,33.0,1794533.514,618749.3427


In [14]:
## beginning of input, any number of digits: r'^\d+'
missed_pickups['building_number'] = missed_pickups.address.str.extract(r'(^\d+)')

In [15]:
missed_pickups.head()

,request_num,date,request,description,address,zip,trash_hauler,trash_route,council_district,state_plan_x,state_plan_y,building_number
1,25274,11/1/2017,Trash - Curbside/Alley Missed Pickup,Curb/Trash miss Tuesday.,4028 Clarksville Pike,37218.0,RED RIVER,4202,1.0,1721259.366,685444.7996,4028
2,25276,11/1/2017,Trash - Curbside/Alley Missed Pickup,Curb/trash miss Tuesday.,6528 Thunderbird Dr,37209.0,RED RIVER,4205,20.0,1707026.753,659887.4716,6528
3,25307,11/1/2017,Trash - Curbside/Alley Missed Pickup,missed,2603 Old Matthews Rd,37207.0,WASTE IND,2206,2.0,1735691.771,685027.2459,2603
4,25312,11/1/2017,Trash - Curbside/Alley Missed Pickup,Missed the even side of the road.,604 Croley Dr,37209.0,RED RIVER,4203,20.0,1710185.772,664205.1011,604
8,25330,11/1/2017,Trash - Curbside/Alley Missed Pickup,Missed.,4484 Lavergne Couchville Pike,37013.0,RED RIVER,4210,33.0,1794533.514,618749.3427,4484


In [16]:
missed_pickups.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17734 entries, 1 to 20205
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   request_num       17734 non-null  int64  
 1   date              17734 non-null  object 
 2   request           17734 non-null  object 
 3   description       17705 non-null  object 
 4   address           17734 non-null  object 
 5   zip               17680 non-null  float64
 6   trash_hauler      16993 non-null  object 
 7   trash_route       16974 non-null  object 
 8   council_district  17700 non-null  float64
 9   state_plan_x      17716 non-null  float64
 10  state_plan_y      17716 non-null  float64
 11  building_number   17695 non-null  object 
dtypes: float64(4), int64(1), object(7)
memory usage: 2.3+ MB


In [17]:
missed_pickups = missed_pickups.dropna(subset=['building_number'])
missed_pickups.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17695 entries, 1 to 20205
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   request_num       17695 non-null  int64  
 1   date              17695 non-null  object 
 2   request           17695 non-null  object 
 3   description       17666 non-null  object 
 4   address           17695 non-null  object 
 5   zip               17645 non-null  float64
 6   trash_hauler      16976 non-null  object 
 7   trash_route       16957 non-null  object 
 8   council_district  17665 non-null  float64
 9   state_plan_x      17680 non-null  float64
 10  state_plan_y      17680 non-null  float64
 11  building_number   17695 non-null  object 
dtypes: float64(4), int64(1), object(7)
memory usage: 1.8+ MB


In [18]:
missed_pickups[missed_pickups.duplicated(subset=['address', 'date'], keep=False)]

,request_num,date,request,description,address,zip,trash_hauler,trash_route,council_district,state_plan_x,state_plan_y,building_number
121,26160,11/6/2017,Trash - Curbside/Alley Missed Pickup,Missed entire area.,2511 Lakevilla Dr,37217.0,RED RIVER,4502,29.0,1778601.178,636324.9504,2511
136,26283,11/6/2017,Trash - Curbside/Alley Missed Pickup,missed,3124 Murfreesboro Pike,37013.0,RED RIVER,4502,33.0,1784737.314,629098.4513,3124
137,26284,11/6/2017,Trash - Curbside/Alley Missed Pickup,missed,3124 Murfreesboro Pike,37013.0,RED RIVER,4502,33.0,1784737.314,629098.4513,3124
146,26378,11/6/2017,Trash - Curbside/Alley Missed Pickup,missed pick up - others missed in area,2511 Lakevilla Dr,37217.0,RED RIVER,4502,29.0,1778601.178,636324.9504,2511
155,26673,11/7/2017,Trash - Curbside/Alley Missed Pickup,customer says trash is not being picked up bet...,111 2Nd Ave N,37201.0,Metro,NaN,19.0,1739543.365,666600.8018,111
...,...,...,...,...,...,...,...,...,...,...,...,...
20101,265938,10/30/2019,Trash - Backdoor,Trash keeps getting miss.,1719 Windover Dr,37218.0,RED RIVER,4202,2.0,1718308.693,680762.9075,1719
20118,266161,10/30/2019,Trash - Backdoor,"Backdoor/trash keeps being missed, Tuesday {wa...",1719 Windover Dr,37218.0,RED RIVER,4202,2.0,1718308.693,680762.9075,1719
20128,266271,10/31/2019,Trash - Backdoor,"missed backdoor pickup, last week and this wee...",4305 Granny White Pike,37204.0,RED RIVER,3302,25.0,1731857.800,644223.2015,4305
20165,266605,10/31/2019,Trash - Backdoor,HAS MISSED BACK DOOR TRASH PICK UP AGAIN/ ALSO...,4305 Granny White Pike,37204.0,RED RIVER,3302,25.0,1731857.800,644223.2015,4305


In [19]:
missed_pickups = missed_pickups.drop_duplicates(subset=['address', 'date'])

In [20]:
missed_pickups.shape

(17325, 12)

In [21]:
missed_pickups.trash_hauler = missed_pickups.trash_hauler.str.upper()

In [22]:
missed_pickups = missed_pickups.dropna(subset=['zip'])
missed_pickups.zip = missed_pickups.zip.astype(int)
missed_pickups.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17276 entries, 1 to 20205
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   request_num       17276 non-null  int64  
 1   date              17276 non-null  object 
 2   request           17276 non-null  object 
 3   description       17249 non-null  object 
 4   address           17276 non-null  object 
 5   zip               17276 non-null  int64  
 6   trash_hauler      16617 non-null  object 
 7   trash_route       16596 non-null  object 
 8   council_district  17275 non-null  float64
 9   state_plan_x      17272 non-null  float64
 10  state_plan_y      17272 non-null  float64
 11  building_number   17276 non-null  object 
dtypes: float64(3), int64(2), object(7)
memory usage: 1.7+ MB


In [23]:
missed_pickups_calc = missed_pickups[['trash_hauler', 'address', 'date']]
missed_pickups_calc.head()

,trash_hauler,address,date
1,RED RIVER,4028 Clarksville Pike,11/1/2017
2,RED RIVER,6528 Thunderbird Dr,11/1/2017
3,WASTE IND,2603 Old Matthews Rd,11/1/2017
4,RED RIVER,604 Croley Dr,11/1/2017
8,RED RIVER,4484 Lavergne Couchville Pike,11/1/2017


In [24]:
missed_pickups_calc.groupby(['trash_hauler', 'address']).count()

date
trash_hauler address                    
METRO        1 Belle Forrest Ave C     1
             10 Belle Forrest Ave      1
             100 Marshall Ct           2
             1000 Gilmore Ave          1
             1000 N 7Th St             1
...                                  ...
WASTE IND    944 4Th Ave S             2
             945 31St Ave N            1
             945 4Th Ave S             2
             951 Sharpe Ave            1
             96 Maury St               1

[11142 rows x 1 columns]

In [25]:
missed_pickups_calc_test = missed_pickups_calc.groupby(['trash_hauler', 'address'], as_index=False).count()

In [26]:
missed_pickups_calc_test = missed_pickups_calc_test.loc[missed_pickups_calc_test.date != 1]
missed_pickups_calc_test.head()

,trash_hauler,address,date
2,METRO,100 Marshall Ct,2
17,METRO,1005 9Th Ave S,2
19,METRO,1005 N 6Th St,2
23,METRO,1006 Pennock Ave,2
37,METRO,1009 14Th Ave S,3


In [27]:
missed_pickups_calc_test['num_of_fines'] = missed_pickups_calc_test.date - 1
missed_pickups_calc_test.head()

,trash_hauler,address,date,num_of_fines
2,METRO,100 Marshall Ct,2,1
17,METRO,1005 9Th Ave S,2,1
19,METRO,1005 N 6Th St,2,1
23,METRO,1006 Pennock Ave,2,1
37,METRO,1009 14Th Ave S,3,2


In [28]:
missed_pickups_calc_test['fines_amount'] = missed_pickups_calc_test.num_of_fines * 200
missed_pickups_calc_test.head()

,trash_hauler,address,date,num_of_fines,fines_amount
2,METRO,100 Marshall Ct,2,1,200
17,METRO,1005 9Th Ave S,2,1,200
19,METRO,1005 N 6Th St,2,1,200
23,METRO,1006 Pennock Ave,2,1,200
37,METRO,1009 14Th Ave S,3,2,400


In [29]:
missed_pickups_calc_test.groupby('trash_hauler', as_index=False).sum()

,trash_hauler,address,date,num_of_fines,fines_amount
0,METRO,100 Marshall Ct1005 9Th Ave S1005 N 6Th St1006...,1529,1003,200600
1,RED RIVER,100 Nashboro Greens100 Rhine Dr1000 Flintlock ...,6319,4130,826000
2,WASTE IND,1002 40Th Ave N1011 Elm Hill Pike1011 Elm Hill...,555,342,68400


In [30]:
missed_calc_zip = missed_pickups[['trash_hauler', 'zip', 'address', 'date']]
missed_calc_zip.head()

,trash_hauler,zip,address,date
1,RED RIVER,37218,4028 Clarksville Pike,11/1/2017
2,RED RIVER,37209,6528 Thunderbird Dr,11/1/2017
3,WASTE IND,37207,2603 Old Matthews Rd,11/1/2017
4,RED RIVER,37209,604 Croley Dr,11/1/2017
8,RED RIVER,37013,4484 Lavergne Couchville Pike,11/1/2017


In [31]:
missed_calc_zip_test = missed_calc_zip.groupby(['trash_hauler', 'zip', 'address'], as_index=False).count()
missed_calc_zip_test.head()

,trash_hauler,zip,address,date
0,METRO,37013,109 Chetopa Ct Antioch Tennessee 37013,1
1,METRO,37201,111 2Nd Ave N,1
2,METRO,37201,117 Union St,1
3,METRO,37201,184 2Nd Ave N,1
4,METRO,37201,200 2Nd Ave S,3


In [32]:
missed_calc_zip_test = missed_calc_zip_test.loc[missed_calc_zip_test.date != 1]
missed_calc_zip_test['num_of_fines'] = missed_calc_zip_test.date - 1
missed_calc_zip_test['fines_amount'] = missed_calc_zip_test.num_of_fines * 200
missed_calc_zip_test.head()

,trash_hauler,zip,address,date,num_of_fines,fines_amount
4,METRO,37201,200 2Nd Ave S,3,2,400
5,METRO,37201,217 2Nd Ave S,6,5,1000
8,METRO,37201,317 Broadway,2,1,200
14,METRO,37203,1005 9Th Ave S,2,1,200
20,METRO,37203,1009 Southside Ave B,2,1,200


In [33]:
missed_calc_zip_test.groupby(['trash_hauler', 'zip'], as_index=False).sum()

,trash_hauler,zip,address,date,num_of_fines,fines_amount
0,METRO,37201,200 2Nd Ave S217 2Nd Ave S317 Broadway,11,8,1600
1,METRO,37203,1005 9Th Ave S1009 Southside Ave B1035 Archer ...,145,101,20200
2,METRO,37204,1014 Jessamin Rd1014 Montrose Ave1016 Montrose...,106,66,13200
3,METRO,37205,130 Wilson Blvd314 Harvard Ave3511 Richland Av...,53,33,6600
4,METRO,37206,1032 Chicamauga Ave1040 Chicamauga Ave1122 Cah...,256,167,33400
5,METRO,37207,1005 N 6Th St1006 Pennock Ave1011 Stockell St1...,191,127,25400
6,METRO,37208,1026 14Th Ave N1212 Cecilia St1217 Ireland St1...,331,218,43600
7,METRO,37209,1101 55Th Ave N1301 54Th Ave N1319 51St Ave N ...,118,86,17200
8,METRO,37210,1029 1St Ave S,2,1,200
9,METRO,37211,196 Chilton St201 Mccall St2122 Utopia Ave215 ...,69,43,8600


In [34]:
missed_calc_zip_test.to_csv('../data/missed_calc_zip_test.csv')

In [39]:
missed_a1_raw = missed_pickups[['trash_hauler', 'address', 'date']]
missed_a1_raw.head()

,trash_hauler,address,date
1,RED RIVER,4028 Clarksville Pike,11/1/2017
2,RED RIVER,6528 Thunderbird Dr,11/1/2017
3,WASTE IND,2603 Old Matthews Rd,11/1/2017
4,RED RIVER,604 Croley Dr,11/1/2017
8,RED RIVER,4484 Lavergne Couchville Pike,11/1/2017


In [62]:
missed_a1_raw.date = pd.to_datetime(missed_a1_raw['date'])
missed_a1_raw.dtypes

C:\Users\erics\AppData\Local\Temp\ipykernel_40192\959537913.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missed_a1_raw.date = pd.to_datetime(missed_a1_raw['date'])


trash_hauler            object
address                 object
date            datetime64[ns]
dtype: object

In [63]:
missed_a1_calc = missed_a1_raw.groupby(['trash_hauler', 'address'], as_index=False).count()
missed_a1_calc = missed_a1_calc.rename(columns = {'date': 'total_missed'})
missed_a1_calc = missed_a1_calc.loc[missed_a1_calc.total_missed > 2]
print(missed_a1_calc.shape)
missed_a1_calc.head()

(1120, 3)


,trash_hauler,address,total_missed
37,METRO,1009 14Th Ave S,3
47,METRO,1011 Stockell St,4
75,METRO,1016 Montrose Ave,3
77,METRO,1016 Stainback Ave,3
104,METRO,1027 Burchwood Ave,3


In [145]:
missed_a1_calc2= missed_a1_raw.merge(missed_a1_calc, left_on=['trash_hauler', 'address'], right_on=['trash_hauler', 'address'],  how='inner').sort_values(by=['trash_hauler', 'address', 'date']).reset_index(drop=True)
print(missed_a1_calc2.shape)
missed_a1_calc2.head(19)

(4787, 4)


,trash_hauler,address,date,total_missed
0,METRO,1009 14Th Ave S,2018-07-23,3
1,METRO,1009 14Th Ave S,2019-03-26,3
2,METRO,1009 14Th Ave S,2019-09-27,3
3,METRO,1011 Stockell St,2018-01-25,4
4,METRO,1011 Stockell St,2018-02-01,4
5,METRO,1011 Stockell St,2018-02-07,4
6,METRO,1011 Stockell St,2018-02-16,4
7,METRO,1016 Montrose Ave,2018-01-11,3
8,METRO,1016 Montrose Ave,2018-02-15,3
9,METRO,1016 Montrose Ave,2018-09-06,3


In [76]:
len(missed_a1_calc2)-1

4786

In [143]:
missed_a1_calc2 = missed_a1_calc2.head(7)

In [149]:
hauler = ''
l_index = len(missed_a1_calc2)-1
c=0
t_miss = 0
address = ''
for index, row in missed_a1_calc2.iterrows():
    if l_index-1 > index:
        if row.trash_hauler != hauler:
            if hauler == '':
                hauler = row.trash_hauler
                address = row.address
                t_miss = row.total_missed
                c = 0
            else:
                print(hauler,':',c*1500)
                hauler = row.trash_hauler
                address = row.address
                t_miss = row.total_missed
                c = 0
            if missed_a1_calc2.loc[index+2, 'date'] - row.date <= timedelta(days=180):
                c+=1
                t_miss-=1
            else:
                t_miss-=1
        elif address == row.address and t_miss > 2:     
            if missed_a1_calc2.loc[index+2, 'date'] - row.date <= timedelta(days=180):
                c+=1
                t_miss-=1
            else:
                t_miss-=1
        else:
            address = row.address
            t_miss = row.total_missed  
            t_miss-=1
    elif l_index > index:
                print(hauler,':',c*1500)

METRO : 448500
RED RIVER : 1830000
WASTE IND : 115500


In [86]:
missed_a1_calc.loc[missed_a1_calc.trash_hauler == 'WASTE IND'].shape

(56, 3)

In [87]:
test = missed_a1_raw.groupby(['trash_hauler', 'address'], as_index=False).count()
test.loc[test.trash_hauler == 'WASTE IND'].shape

(759, 3)

In [159]:
q1a = trash_hauler_report.request.value_counts().reset_index().rename(columns = {'count': 'total'})

In [160]:
q1b = missed2.request.value_counts().reset_index().rename(columns = {'count': 'missed_pickup'})

In [163]:
q1 = q1a.merge(q1b, left_on='request', right_on='request',  how='inner')
q1['diff'] = q1.total - q1.missed_pickup
q1.to_csv('../data/q1.csv')
q1

,request,total,missed_pickup,diff
0,Trash - Curbside/Alley Missed Pickup,15028,10329,4699
1,Trash - Backdoor,2629,2017,612
2,Trash Collection Complaint,2312,695,1617
3,Damage to Property,257,3,254


In [ ]:
hauler = ''
l_index = len(missed_a1_calc2)-1
c=0
s=0
for index, row in missed_a1_calc2.iterrows():
    if l_index-1 > index and s == 0:
        if row.trash_hauler != hauler:
            if hauler == '':
                hauler = row.trash_hauler
                address = row.address
                t_miss = row.total_missed
                c = 0
            else:
                print(hauler,':',c*1500)
                hauler = row.trash_hauler
                address = row.address
                t_miss = row.total_missed
                c = 0
            if missed_a1_calc2.loc[index+2, 'date'] - row.date >= timedelta(days=180):
                c+=1
                s+=2
                t_miss-=1
            else:
                t_miss-=1
        elif address == row.address and t_miss > 2:     
            if missed_a1_calc2.loc[index+2, 'date'] - row.date >= timedelta(days=180):
                c+=1
                s+=2
                t_miss-=1
            else:
                t_miss-=1
        else:
            address = row.address
            t_miss = row.total_missed  
            if missed_a1_calc2.loc[index+2, 'date'] - row.date >= timedelta(days=180):
                c+=1
                s+=2
                t_miss-=1
            else:
                t_miss-=1
    elif l_index > index:
                print(hauler,':',c*1500)